# 🔗 Notebook 3: ORNJ Relation Extraction

Extract classification criteria, staging logic, and treatment relations.

---

In [ ]:
!pip install -q PyMuPDF pdfplumber pytesseract Pillow spacy scikit-learn owlready2 rdflib networkx matplotlib anthropic tqdm
!apt-get install -q tesseract-ocr > /dev/null 2>&1
!python -m spacy download en_core_web_sm -q
!git clone https://github.com/YOUR_USERNAME/orn-ontology-builder.git 2>/dev/null || true
import sys; sys.path.insert(0, 'orn-ontology-builder')

## 1. Load Data

In [ ]:
import json, os
from src.preprocessor import TextPreprocessor
from src.concept_extractor import ConceptExtractor

# Demo text: ORNJ classification systems (use when no PDFs uploaded)
demo_texts = [
    """Notani et al. divided cases of mandibular osteoradionecrosis into
    three grades based on the extent of the lesion. Grade I is defined
    as ORN confined to the alveolar bone. Grade II is defined as ORN
    limited to the alveolar bone and/or the mandible above the level
    of the inferior alveolar canal. Grade III is ORN extending below
    the inferior alveolar canal with or without pathological fracture
    or orocutaneous fistula. Treatments such as sequestrectomy,
    marginal resection, and segmental resection with free flap
    reconstruction are recommended based on grade severity.""",

    """The ClinRad classification system proposed by Watson et al. in 2024
    considers both clinical and radiographic features to stage ORNJ.
    Grade 0 represents minor bone spicules without clinical significance.
    Grade 1 involves alveolar bone necrosis detected radiographically
    without bone exposure. Grade 2 involves alveolar bone necrosis with
    exposed bone or probe-to-bone positive findings. Grade 3 involves
    basilar bone involvement or maxillary sinus involvement. Grade 4
    involves pathological fracture, orocutaneous fistula, or extensive
    bone destruction. The ClinRad system was endorsed by ISOO-MASCC-ASCO.""",

    """Marx proposed a three-stage system for osteoradionecrosis in 1983.
    Stage I patients exhibit exposed bone that has failed to heal for
    at least 6 months and receive 30 sessions of hyperbaric oxygen
    therapy. Stage I responders show tissue softening and spontaneous
    sequestration. Stage II involves non-responsive cases requiring
    surgery combined with HBO. Stage III involves pathological fracture
    or orocutaneous fistula requiring mandibular resection and free flap
    reconstruction. Risk factors include radiation dose exceeding 60 Gy,
    smoking, dental extraction after radiotherapy, and stage III-IV
    periodontal disease.""",
]

pp = TextPreprocessor(min_segment_length=10)
segments = []
for t in demo_texts:
    segments.extend(pp.preprocess(t))
ce = ConceptExtractor(min_frequency=1)
concepts = ce.extract(segments)
print(f'{len(segments)} segments, {len(concepts)} concepts')

## 2. Hearst Pattern Extraction

In [ ]:
import re
test = 'Treatments such as HBO, PENTOCLO, and sequestrectomy are used for ORN.'
pattern = r'(\b[A-Z][a-z]+(?:\s+[a-z]+)*)\s+such\s+as\s+((?:[A-Z][a-z]+(?:\s+[a-z]+)*(?:,\s*)?)+(?:\s+and\s+[A-Z][a-z]+(?:\s+[a-z]+)*)?)'
for m in re.finditer(pattern, test):
    hypernym = m.group(1)
    hyponyms = re.split(r',\s*|\s+and\s+', m.group(2))
    for h in hyponyms:
        if h.strip():
            print(f'  {h.strip()} ──isA──▶ {hypernym}')

## 3. Full Relation Extraction

In [ ]:
from src.relation_extractor import RelationExtractor
rel_ext = RelationExtractor(use_hearst=True, use_dependency=True, min_confidence=0.0)
relations = rel_ext.extract(segments, concepts)
print(f'\nExtracted {len(relations)} relations:')
for r in relations[:20]:
    print(f'  {r.subject:<25} ──{r.predicate}──▶ {r.object}  (conf={r.confidence:.2f}, {r.relation_type})')

## 4. Visualize

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
G = nx.DiGraph()
for r in relations:
    G.add_edge(r.subject, r.object, label=r.predicate)
if G.nodes:
    plt.figure(figsize=(14, 10))
    pos = nx.spring_layout(G, k=2, seed=42)
    nx.draw_networkx_nodes(G, pos, node_size=500, node_color='#E6F1FB', edgecolors='#378ADD')
    nx.draw_networkx_edges(G, pos, arrows=True, arrowsize=12, alpha=0.6)
    nx.draw_networkx_labels(G, pos, font_size=7, font_weight='bold')
    edge_labels = {(u,v): d['label'] for u,v,d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=5)
    plt.title('ORNJ Classification Relations', fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## 5. Save

In [ ]:
os.makedirs('output', exist_ok=True)
with open('output/relations.json', 'w') as f:
    json.dump([{'subject': r.subject, 'predicate': r.predicate, 'object': r.object, 'confidence': r.confidence, 'type': r.relation_type} for r in relations], f, indent=2)
print(f'Saved {len(relations)} relations')

---
**Next:** [Notebook 4 — Ontology Building](04_ontology_building.ipynb)